In [2]:
import pandas as pd
import numpy as np


In [4]:

# 1. Load dataset
df = pd.read_csv("cleaned_city_day.csv")

# 2. Convert Date to datetime format
df["Date"] = pd.to_datetime(df["Date"])

# 3. Sort data city-wise and date-wise
df = df.sort_values(["City", "Date"]).reset_index(drop=True)

# 4. Create date features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek

# 5. Create previous AQI features for each city
group = df.groupby("City")["AQI"]

df["AQI_lag_1"] = group.shift(1)
df["AQI_lag_2"] = group.shift(2)
df["AQI_lag_3"] = group.shift(3)

# 6. Target: next recorded AQI for the same city
df["Target_AQI"] = group.shift(-1)
df["Target_Date"] = df.groupby("City")["Date"].shift(-1)

# 7. Remove rows without required lag values or target
df = df.dropna(
    subset=[
        "AQI_lag_1",
        "AQI_lag_2",
        "AQI_lag_3",
        "Target_AQI",
        "Target_Date"
    ]
).copy()

# 8. Select input features (X)
features = [
    "City",
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3", "Benzene", "Toluene",
    "Year", "Month", "Day", "DayOfWeek",
    "AQI_lag_1", "AQI_lag_2", "AQI_lag_3"
]


In [5]:
X = df[features]

# 9. Select target (y)
y = df["Target_AQI"]

# 10. Chronological 80/20 split
unique_dates = np.sort(df["Date"].unique())
split_index = int(len(unique_dates) * 0.8)
split_date = pd.Timestamp(unique_dates[split_index])

# Training: both observation and target must be before cutoff
train_mask = (
    (df["Date"] < split_date) &
    (df["Target_Date"] < split_date)
)

# Testing: observations and their targets are in the test period
test_mask = (
    (df["Date"] >= split_date) &
    (df["Target_Date"] >= split_date)
)
X = pd.get_dummies(X, columns=["City"], dtype=int)
X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

# 11. Check the results
print("Split date:", split_date.date())
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining date range:")
print(df.loc[train_mask, "Date"].min(), "to",
      df.loc[train_mask, "Date"].max())

print("\nTesting date range:")
print(df.loc[test_mask, "Date"].min(), "to",
      df.loc[test_mask, "Date"].max())

Split date: 2019-05-27
X_train shape: (15942, 44)
X_test shape: (8785, 44)
y_train shape: (15942,)
y_test shape: (8785,)

Training date range:
2015-01-04 00:00:00 to 2019-05-25 00:00:00

Testing date range:
2019-05-27 00:00:00 to 2020-06-30 00:00:00


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
l_model = LinearRegression();
l_model.fit(X_train, y_train)
y_pred_l = l_model.predict(X_test)
print(r2_score(y_test, y_pred_l))
print("MAE:", mean_absolute_error(y_test, y_pred_l))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_l)))

0.8313987422039884
MAE: 25.333039291963964
RMSE: 42.75389014654066


In [8]:
from sklearn.ensemble import RandomForestRegressor
r_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)
r_model.fit(X_train, y_train)
y_pred_r = r_model.predict(X_test)
print(r2_score(y_test, y_pred_r))
print("MAE:", mean_absolute_error(y_test, y_pred_r))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_r)))


0.8212916265456781
MAE: 21.94131119620981
RMSE: 44.016720595750144


In [ ]:
from sklearn.tree import DecisionTreeRegressor
d_model = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=2,
    random_state=42
)
d_model.fit(X_train, y_train)
y_pred_d = d_model.predict(X_test)
print(r2_score(y_test, y_pred_d))
print("MAE:", mean_absolute_error(y_test, y_pred_d))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_d)))

0.7322780897948005


In [ ]:

from sklearn.preprocessing import StandardScaler

# Create a scaler
scaler = StandardScaler()

# Create NEW scaled datasets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Original training data:", X_train.shape)
print("Scaled training data:", X_train_scaled.shape)

print("Original test data:", X_test.shape)
print("Scaled test data:", X_test_scaled.shape)



from sklearn.svm import LinearSVR
from sklearn.metrics import mean_absolute_error, mean_squared_error

svm_model = LinearSVR(
    C=1.0,
    max_iter=5000,
    random_state=42,
    dual="auto"
)

svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)

print("R2:", r2_score(y_test, y_pred_svm))


Original training data: (15942, 44)
Scaled training data: (15942, 44)
Original test data: (8785, 44)
Scaled test data: (8785, 44)
R2: 0.8385236311193456
MAE: 22.598264971699084
RMSE: 41.84077491725535


In [51]:

import numpy as np
import pandas as pd

# Combine predictions into 5 input columns
X_meta = np.column_stack([
    np.asarray(y_pred_l).ravel(),
    np.asarray(y_pred_r).ravel(),
    np.asarray(y_pred_d).ravel(),
    np.asarray(y_pred_svm).ravel(),
])

# Actual AQI is the target
y_meta = np.asarray(y_test).ravel()

print("Neural network input shape:", X_meta.shape)
print("Actual AQI shape:", y_meta.shape)


Neural network input shape: (8785, 4)
Actual AQI shape: (8785,)


In [ ]:

# Do NOT shuffle time-series data
split_idx = int(len(X_meta) * 0.8)

X_meta_train = X_meta[:split_idx]
X_meta_test = X_meta[split_idx:]

y_meta_train = y_meta[:split_idx]
y_meta_test = y_meta[split_idx:]

print("Meta training:", X_meta_train.shape)
print("Meta testing:", X_meta_test.shape)

meta_scaler = StandardScaler()

X_meta_train_scaled = meta_scaler.fit_transform(X_meta_train)
X_meta_test_scaled = meta_scaler.transform(X_meta_test)

Meta training: (7028, 4)
Meta testing: (1757, 4)


In [53]:

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

nn_model = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16, 8, 4),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=1000,
    early_stopping=True,
    random_state=42
)


nn_model.fit(X_meta_train_scaled, y_meta_train)


y_pred_nn = nn_model.predict(X_meta_test_scaled)


print("Neural Network Ensemble Results")
print("R2:", r2_score(y_meta_test, y_pred_nn))
print("MAE:", mean_absolute_error(y_meta_test, y_pred_nn))
print("RMSE:", np.sqrt(mean_squared_error(y_meta_test, y_pred_nn)))

print("Final training loss:", nn_model.loss_)
print("Iterations used:", nn_model.n_iter_)

Neural Network Ensemble Results
R2: 0.9021200649889606
MAE: 16.304884110492015
RMSE: 25.30053690050004
Final training loss: 844.1892423676614
Iterations used: 104


In [54]:
import joblib
import os

os.makedirs("saved_models", exist_ok=True)

models = {
    "linear_regression": l_model,
    "random_forest": r_model,
    "decision_tree": d_model,
    "linear_svr": svm_model,
    "neural_network_ensemble": nn_model
}

for name, model in models.items():
    joblib.dump(model, f"saved_models/{name}.joblib")

print("All models saved successfully!")

All models saved successfully!


In [55]:
joblib.dump(scaler, "saved_models/svr_scaler.joblib")

# If your neural network used a scaler on its four input predictions:
joblib.dump(meta_scaler, "saved_models/meta_scaler.joblib")

['saved_models/meta_scaler.joblib']

In [56]:
feature_columns = X_train.columns.tolist()

joblib.dump(
    feature_columns,
    "saved_models/feature_columns.joblib"
)

['saved_models/feature_columns.joblib']